# Pipeline CCCD — Python 3.12.3, giữ model giữa các lần Run trên Colab

Kernel giao diện Colab có thể dùng Python 3.13. Toàn bộ VN-ID, Torch, QR và VietOCR chạy trong một process Python **3.12.3** từ `/content/venv312`.

1. Chạy các cell chuẩn bị môi trường/model theo thứ tự.
2. Chạy mục 6 lần đầu: khởi động process, import, load model và xử lý ảnh đã chọn.
3. Bấm Run lại **riêng cell mục 6**: gửi yêu cầu vào cùng process 3.12.3 và tái sử dụng model.

`total_ms` là thời gian nội bộ pipeline. `pipeline_wall_ms` đo lời gọi pipeline có đồng bộ GPU. `cell_wall_ms` đo toàn bộ yêu cầu, gồm preview, in kết quả và JSON. Không cộng ba số này với nhau.

Process mất khi restart/disconnect runtime. Muốn khởi tạo lại hoặc sau khi sửa mã nguồn, chạy `vnid_worker.terminate(); vnid_worker.wait()` rồi chạy mục 6 lại. Các thư viện OCR không được import vào kernel 3.13.


## 1. Tạo môi trường Python 3.12.3


In [ ]:
import os
from pathlib import Path

# Không xóa venv khi chạy lại cell chuẩn bị.
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!uv python install 3.12.3

PY = "/content/venv312/bin/python"
if not Path(PY).exists():
    !uv venv --seed /content/venv312 --python 3.12.3

!{PY} -c "import sys; print(sys.version); assert sys.version_info[:3] == (3, 12, 3)"
!{PY} -m pip --version


## 2. Clone VN-ID-Demo và cài dependency


In [ ]:
import os
import sys

REPO_DIR = "/content/VN-ID-Demo"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Shinoaki0145/VN_ID.git {REPO_DIR}
else:
    print("Repo đã tồn tại:", REPO_DIR)

# System dependency
!apt-get update -qq
!apt-get install -y -qq libzbar0

PY = "/content/venv312/bin/python"

# Build tools
!{PY} -m pip install -U pip setuptools wheel

# PyTorch + các package OCR/QR
!{PY} -m pip install \
    torch torchvision \
    easyocr \
    qreader \
    zxing-cpp \
    pyzbar \
    einops \
    lmdb \
    gdown \
    prefetch-generator \
    opencv-python-headless \
    matplotlib \
    pillow \
    pyyaml \
    numpy \
    qrdet \
    "pydantic>=2,<3"

# VietOCR
!{PY} -m pip install --no-deps git+https://github.com/pbcquoc/vietocr.git

# Nếu repo có requirements.txt thì cài thêm dependency mà tác giả khai báo
if os.path.exists(os.path.join(REPO_DIR, "requirements.txt")):
    !{PY} -m pip install -r {REPO_DIR}/requirements.txt

print("Cài dependency xong.")


In [ ]:
PY = "/content/venv312/bin/python"

# Cài compiler + Ninja cho PyTorch C++/CUDA extensions
!apt-get update -qq
!apt-get install -y build-essential ninja-build

!{PY} -m pip install -U ninja

# Kiểm tra
!{PY} -c "import ninja; print('Ninja:', ninja.__version__)"

## 3. Kiểm tra Python 3.12.3 và GPU của môi trường pipeline


In [ ]:
%%script /content/venv312/bin/python
import sys
import torch
import cv2
import pydantic

assert sys.version_info[:3] == (3, 12, 3), sys.version
print("Python   :", sys.version)
print("Exe      :", sys.executable)
print("Torch    :", torch.__version__)
print("OpenCV   :", cv2.__version__)
print("Pydantic :", pydantic.__version__)
print("CUDA     :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU      :", torch.cuda.get_device_name(0))


## 4. Chọn ảnh đầu vào — mặt trước, mặt sau hoặc cả hai

Upload ảnh vào Files của Colab rồi đặt đường dẫn tương ứng (ví dụ `/content/front.jpg` hoặc `/content/back.jpg`).

- Chỉ mặt trước: `FRONT_IMAGE` có đường dẫn, `BACK_IMAGE = ""`.
- Chỉ mặt sau: `FRONT_IMAGE = ""`, `BACK_IMAGE` có đường dẫn.
- Cả hai mặt: điền cả hai đường dẫn.
- Không có ảnh nào: báo lỗi, không chạy pipeline.

Chạy cell chọn ảnh, cell ghi script mục 5 nếu vừa cập nhật notebook, rồi cell chạy mục 6. Đổi ảnh vẫn dùng cùng process/model Python 3.12.3.


In [ ]:
# Điền đường dẫn ảnh; để trống mặt không có.
FRONT_IMAGE = "image/front_cu.jpg"
BACK_IMAGE = ""

print("Front:", FRONT_IMAGE or "Không có")
print("Back :", BACK_IMAGE or "Không có")


## 5. Ghi script chạy trong process Python 3.12.3 được giữ lại

Cell này chỉ ghi file. Mục 6 gửi yêu cầu chạy script vào cùng process để giữ biến `pipeline` và model giữa các lần Run.


In [ ]:
%%writefile /content/run_vnid_pipeline.py
import os
import sys
import json

assert sys.version_info[:3] == (3, 12, 3), sys.version
import time
import torch

# Fix backend Matplotlib khi chạy ngoài kernel Colab
os.environ["MPLBACKEND"] = "Agg"

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO_DIR = "/content/VN-ID-Demo"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from vn_id import CCCDPipeline, CardSide

FRONT_IMAGE = os.environ.get("FRONT_IMAGE", "").strip()
BACK_IMAGE = os.environ.get("BACK_IMAGE", "").strip()

OUTPUT_DIR = "/content/vnid_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("Python     :", sys.version)
print("Executable :", sys.executable)
print("Repo       :", REPO_DIR)
print("Front      :", FRONT_IMAGE)
print("Back       :", BACK_IMAGE)
print("=" * 70)


# ------------------------------------------------------------
# Kiểm tra ảnh
# ------------------------------------------------------------
if not FRONT_IMAGE and not BACK_IMAGE:
    raise ValueError("Cần cung cấp ít nhất một ảnh mặt trước hoặc mặt sau.")

# Đọc và tạo preview chỉ cho ảnh được cung cấp trong lượt này.
img_front = None
img_back = None
previews = []
for side_label, image_path in [("Mặt trước", FRONT_IMAGE), ("Mặt sau", BACK_IMAGE)]:
    if not image_path:
        continue
    if not os.path.isfile(image_path):
        raise FileNotFoundError(f"Không tìm thấy: {image_path}")
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"OpenCV không đọc được: {image_path}")
    previews.append((img, f"{side_label}: {image_path}"))
    if side_label == "Mặt trước":
        img_front = img
    else:
        img_back = img

preview_file = os.path.join(OUTPUT_DIR, "input_preview.png")
fig, axes = plt.subplots(1, len(previews), figsize=(6 * len(previews), 5), squeeze=False)
for ax, (img, title) in zip(axes[0], previews):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.savefig(preview_file, dpi=150, bbox_inches="tight")
plt.close(fig)

# ------------------------------------------------------------
# Pipeline chính
# ------------------------------------------------------------
# Process 3.12.3 giữ namespace này giữa các yêu cầu.
if "pipeline" not in globals():
    pipeline = CCCDPipeline(device="auto", mock_mode=False)
    pipeline_run_count = 0

pipeline_run_count = globals().get("pipeline_run_count", 0) + 1
print("Pipeline device:", pipeline.device)
print("Lượt trong process 3.12.3:", pipeline_run_count)
print("Lần đầu có chi phí khởi tạo; các lượt sau tái sử dụng model.")

if pipeline.device.startswith("cuda"):
    torch.cuda.synchronize()
pipeline_start = time.perf_counter()
if FRONT_IMAGE and BACK_IMAGE:
    result = pipeline.process_both_sides(
        front_image=FRONT_IMAGE,
        back_image=BACK_IMAGE,
    )
elif FRONT_IMAGE:
    result = pipeline.process(image=FRONT_IMAGE, force_side=CardSide.FRONT)
else:
    result = pipeline.process(image=BACK_IMAGE, force_side=CardSide.BACK)
if pipeline.device.startswith("cuda"):
    torch.cuda.synchronize()
pipeline_wall_ms = (time.perf_counter() - pipeline_start) * 1000
print(f"pipeline_wall_ms (đồng bộ CUDA nếu có): {pipeline_wall_ms:.2f} ms")

print("\n=== KẾT QUẢ TRÍCH XUẤT THẬT ===")
print("Thành công:", result.success)
print("Mặt phát hiện:", result.side_detected)
print("Nguồn dữ liệu từng trường:", result.field_sources)

print("\n=== ĐO ĐỘ TRỄ (LATENCY BREAKDOWN) ===")
for step, ms in result.timings.items():
    print(f"  * {step:<18}: {ms:8.2f} ms")
print("-" * 36)
print(f"  * TỔNG THỜI GIAN    : {result.timings.get('total_ms', 0):8.2f} ms")

# ------------------------------------------------------------
# Chi tiết dữ liệu
# ------------------------------------------------------------
data = result.data
val = result.validation

print("\n+" + "-"*28 + "+" + "-"*52 + "+")
print(f"| {'TRƯỜNG THÔNG TIN':<26} | {'GIÁ TRỊ':<50} |")
print("+" + "-"*28 + "+" + "-"*52 + "+")

fields = [
    ("Số CCCD", "id"),
    ("Số CMND cũ", "cmnd_old"),
    ("Họ và tên", "name"),
    ("Ngày sinh", "dob"),
    ("Giới tính", "gender"),
    ("Quốc tịch", "nationality"),
    ("Quê quán / Nơi ĐKKS", "origin"),
    ("Nơi thường trú / Cư trú", "residence"),
    ("Ngày cấp", "issue_date"),
    ("Nơi cấp", "issue_loc"),
    ("Có giá trị đến", "expiry_date"),
]

for label, attr in fields:
    value = getattr(data, attr, None)
    print(f"| {label:<26} | {str(value):<50} |")

print("+" + "-"*28 + "+" + "-"*52 + "+")

print("\nĐối soát CCCD:")
print("12 số    :", getattr(val, "is_valid_12_digits", None))
print("Tỉnh     :", getattr(val, "province_valid", None))
print("Năm sinh :", getattr(val, "birth_year_valid", None))

# ------------------------------------------------------------
# Xuất JSON
# ------------------------------------------------------------
output_file = os.path.join(OUTPUT_DIR, "output_result.json")

if hasattr(result, "model_dump"):
    payload = result.model_dump()
elif hasattr(result, "dict"):
    payload = result.dict()
else:
    raise TypeError("Không thể serialize result sang JSON")

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2, default=str)

print("\n=== HOÀN TẤT ===")
print("JSON   :", output_file)
print("Preview:", preview_file)


In [ ]:
# ============================================================
# PATCH EASYOCR DBNET CUDA CHO PYTORCH MỚI
# ============================================================

from pathlib import Path
import subprocess
import shutil

# Hỏi Python 3.12.3 vị trí easyocr, không import package vào kernel 3.13.
PY = "/content/venv312/bin/python"
easyocr_origin = subprocess.check_output(
    [PY, "-c", "import importlib.util; print(importlib.util.find_spec('easyocr').origin)"],
    text=True,
).strip()
DCN_DIR = Path(easyocr_origin).parent / "DBNet/assets/ops/dcn/src"

files = [
    DCN_DIR / "deform_conv_cuda_kernel.cu",
    DCN_DIR / "deform_pool_cuda_kernel.cu",
]

for file in files:
    print(f"\nPatching: {file}")

    # backup
    backup = Path(str(file) + ".bak")
    if not backup.exists():
        shutil.copy2(file, backup)

    text = file.read_text()

    before = text.count(".type()")

    # PyTorch cũ:
    # tensor.type()
    #
    # PyTorch mới:
    # tensor.scalar_type()
    text = text.replace(".type()", ".scalar_type()")

    file.write_text(text)

    after = text.count(".type()")

    print("  .type() trước patch :", before)
    print("  .type() sau patch   :", after)

print("\nPatch hoàn tất.")

In [ ]:
# Kiểm tra patch trong package của kernel hiện tại.
for file in files:
    remaining = file.read_text().count(".type()")
    print(file.name, "số .type() còn lại:", remaining)
    assert remaining == 0


In [ ]:
PY = "/content/venv312/bin/python"

!{PY} -m pip install -U ninja
!{PY} -c "import ninja; print('Ninja:', ninja.__version__)"

In [ ]:
from pathlib import Path

yaml_path = Path("/content/VN-ID-Demo/vn_id/ocr/vgg_seq2seq.yml")

text = yaml_path.read_text(encoding="utf-8")

text = text.replace(
    "weights: /home/thiennhan/IDX/CCCD/VN_ID/.easyocr_models/model/vgg_seq2seq.pth",
    "weights: /content/VN-ID-Demo/.easyocr_models/model/vgg_seq2seq.pth"
)

yaml_path.write_text(text, encoding="utf-8")

print(yaml_path.read_text(encoding="utf-8"))

In [ ]:
!mkdir -p /content/VN-ID-Demo/.easyocr_models/model

In [ ]:
!wget -O /content/VN-ID-Demo/.easyocr_models/model/vgg_seq2seq.pth \
https://vocr.vn/data/vietocr/vgg_seq2seq.pth

In [ ]:
!ls -lh /content/VN-ID-Demo/.easyocr_models/model/vgg_seq2seq.pth

## 6. Chạy pipeline Python 3.12.3 — giữ process và model giữa các lần Run

Bấm Run lần đầu để đo khởi tạo. Bấm lại **riêng cell này** để đo lượt tiếp theo khi model đã sẵn sàng. Mỗi lượt chỉ xử lý ảnh đã chọn.

- `total_ms`: thời gian đo nội bộ pipeline.
- `pipeline_wall_ms`: lời gọi pipeline có đồng bộ GPU trước/sau.
- `cell_wall_ms`: toàn bộ yêu cầu, bao gồm các việc ngoài pipeline và truyền output về Colab.

Đổi ảnh ở mục 4 rồi chạy lại cell này. Nếu sửa mã nguồn, dừng process bằng `vnid_worker.terminate(); vnid_worker.wait()` trước khi chạy lại. Nếu Stop/interrupt cell đang chạy, cell sẽ dừng process để tránh đọc nhầm kết quả ở lượt sau.


In [ ]:
import json
import subprocess
import time
import uuid

cell_start = time.perf_counter()
PY = "/content/venv312/bin/python"

# Kernel Colab chỉ dùng stdlib để gửi yêu cầu; mọi OCR/QR chạy trong PY.
worker_code = r"""
import json
import os
import sys
import traceback
from pathlib import Path

assert sys.version_info[:3] == (3, 12, 3), sys.version
scope = {"__name__": "__main__"}
for line in sys.stdin:
    request = json.loads(line)
    ok = False
    try:
        os.environ.update(request["env"])
        script = Path(request["script"])
        scope["__file__"] = str(script)
        exec(compile(script.read_text(encoding="utf-8"), str(script), "exec"), scope)
        ok = True
    except BaseException:
        traceback.print_exc()
    finally:
        print("\n" + request["token"] + json.dumps({"ok": ok}), flush=True)
"""

if "vnid_worker" not in globals() or vnid_worker.poll() is not None:
    vnid_worker = subprocess.Popen(
        [PY, "-u", "-c", worker_code],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    print("Đã tạo process Python 3.12.3, PID:", vnid_worker.pid)
else:
    print("Dùng lại process Python 3.12.3, PID:", vnid_worker.pid)

token = "VNID_DONE_" + uuid.uuid4().hex + ":"
request = {
    "script": "/content/run_vnid_pipeline.py",
    "env": {"FRONT_IMAGE": FRONT_IMAGE or "", "BACK_IMAGE": BACK_IMAGE or "", "MPLBACKEND": "Agg"},
    "token": token,
}
try:
    vnid_worker.stdin.write(json.dumps(request) + "\n")
    vnid_worker.stdin.flush()
    for line in vnid_worker.stdout:
        if line.startswith(token):
            response = json.loads(line[len(token):])
            if not response["ok"]:
                raise RuntimeError("Pipeline bị lỗi; xem traceback phía trên.")
            break
        print(line, end="", flush=True)
    else:
        raise RuntimeError("Process Python 3.12.3 đã kết thúc trước khi trả kết quả.")
except KeyboardInterrupt:
    vnid_worker.terminate()
    vnid_worker.wait()
    raise

cell_wall_ms = (time.perf_counter() - cell_start) * 1000
print(f"\ncell_wall_ms (toàn bộ yêu cầu): {cell_wall_ms:.2f} ms")


## 7. Xem ảnh đầu vào đã render


In [ ]:
from IPython.display import Image, display
import os

preview = "/content/vnid_output/input_preview.png"

if os.path.exists(preview):
    display(Image(filename=preview))
else:
    print("Chưa có preview. Hãy chạy cell ở mục 6 trước.")


## 8. Xem JSON kết quả


In [ ]:
import json
import os

json_path = "/content/vnid_output/output_result.json"

if os.path.exists(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        result_json = json.load(f)
    print(json.dumps(result_json, ensure_ascii=False, indent=2))
else:
    print("Chưa có JSON. Hãy chạy cell ở mục 6 trước.")
